In [6]:
"""
IRASA analysis time-locked to POINTING_FINISHED events.
Experiments: DBOY1, EFRCourierReadOnly
Regions: LHP, RHP only
Window: 0 to +1000 ms post-event
Output: one row per subject × session × pointing_event × region × frequency

Parallelized across subjects using joblib (n_jobs=-1).
"""

import os
import pandas as pd
import numpy as np
import cmlreaders as cml
from joblib import Parallel, delayed

# ============================================================================
# IRASA IMPORTS
# ============================================================================
try:
    from irasa.IRASA import IRASA, SSL_transform
    IRASA_AVAILABLE = True
    print("✓ IRASA imported successfully")
except ImportError as e:
    IRASA_AVAILABLE = False
    print(f"✗ IRASA not available: {e}")
    def SSL_transform(x):
        return np.sign(x) * np.log10(1 + np.abs(x))

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly']

# IRASA parameters (unchanged from original)
IRASA_HSET  = np.arange(1.1, 1.9, 0.05)
IRASA_FREQS = np.round(np.logspace(np.log10(3), np.log10(128), 18), 2)

# Hippocampus only
REGIONS = ['LHP', 'RHP']

# Epoch: load a buffer around the event, then trim to analysis window
# Buffer of 500 ms on each side to absorb edge artefacts from IRASA
EPOCH_START_MS  = -8000  # ms relative to POINTING_FINISHED
EPOCH_END_MS    = -50   # ms relative to POINTING_FINISHED
WIN_START_MS    = -6000  # analysis window start (ms)
WIN_END_MS      = -1000   # analysis window end   (ms)

# Event type to lock to – adjust if the actual label differs
POINTING_EVENT_TYPE = 'REC_START'

OUTPUT_DIR = './irasa_pointing_finished_hp'

N_JOBS = -1   # use all available CPUs

# ============================================================================
# HELPERS
# ============================================================================

def ssl(x):
    return SSL_transform(x)

def ms_to_sample(ms, epoch_start_ms, sfreq):
    return int(round((ms - epoch_start_ms) / 1000.0 * sfreq))

def define_hp_masks(pairs_df):
    """Return LHP / RHP boolean masks, trying multiple region columns."""
    for col in ['dk.region', 'mni.region', 'ind.region']:
        if col in pairs_df.columns:
            region_col = col
            break
    else:
        candidates = [c for c in pairs_df.columns if 'region' in c.lower()]
        if candidates:
            region_col = candidates[0]
            print(f"    WARNING: using {region_col} for region labels")
        else:
            print("    ERROR: no region column found – returning empty masks")
            return {r: np.zeros(len(pairs_df), dtype=bool) for r in REGIONS}

    masks = {
        'LHP': pairs_df[region_col].str.contains('Left.*Hippocampus',  case=False, na=False, regex=True),
        'RHP': pairs_df[region_col].str.contains('Right.*Hippocampus', case=False, na=False, regex=True),
    }
    return masks

def compute_irasa_power_trial(eeg_1d, sfreq, freqs, hset):
    """IRASA on a single 1-D epoch (one channel, one event)."""
    ir = IRASA(
        sig=eeg_1d.astype(float),
        freqs=freqs,
        samplerate=sfreq,
        hset=hset,
        flag_filter=1,
        flag_detrend=1
    )
    mixed   = ir.mixed
    fractal = ir.fractal
    osc     = mixed - fractal
    return mixed, fractal, osc

def irasa_for_event(eeg_window, ch_indices, sfreq):
    """
    Compute SSL-transformed IRASA across all channels in ch_indices for one
    event epoch (eeg_window shape: n_channels × n_samples).

    Returns
    -------
    frac_ssl_mean : array (n_freqs,)  – mean across channels after SSL
    osc_ssl_mean  : array (n_freqs,)
    n_valid_ch    : int
    """
    frac_list = []
    osc_list  = []

    for ch_idx in ch_indices:
        sig = eeg_window[ch_idx, :]
        try:
            _, fractal, osc = compute_irasa_power_trial(sig, sfreq, IRASA_FREQS, IRASA_HSET)
            frac_list.append(ssl(fractal))
            osc_list.append(ssl(osc))
        except Exception:
            pass  # skip noisy / bad channels

    if len(frac_list) == 0:
        nan_vec = np.full(len(IRASA_FREQS), np.nan)
        return nan_vec, nan_vec, 0

    frac_ssl_mean = np.mean(frac_list, axis=0)
    osc_ssl_mean  = np.mean(osc_list,  axis=0)
    return frac_ssl_mean, osc_ssl_mean, len(frac_list)

# ============================================================================
# PER-SUBJECT WORKER  (called in parallel)
# ============================================================================

def process_subject(subject, exp, exp_df, output_dir):
    """
    Process all sessions for one subject in one experiment.

    Returns a list of row-dicts (may be empty) and prints progress to stdout.
    Each worker writes its own per-session CSVs directly to disk so partial
    results are preserved even if the master aggregation fails.
    """
    sub_df   = exp_df.query('subject == @subject')
    sessions = sub_df['session'].unique()

    subject_rows = []
    pairs_df     = None
    region_masks = None

    for session in sessions:
        print(f"  [{exp}] {subject} | session {session}")
        session_rows = []

        # ---- skip already-completed sessions --------------------------------
        fname = os.path.join(output_dir, f'{exp}_{subject}_{session}_pointing_hp_irasa.csv')
        if os.path.exists(fname):
            print(f"    ↩ Already exists, skipping: {fname}")
            try:
                df_existing = pd.read_csv(fname)
                subject_rows.extend(df_existing.to_dict('records'))
            except Exception:
                pass
            continue

        try:
            reader = cml.CMLReader(subject, exp, session=session)

            # Load pairs once per subject (same across sessions)
            if pairs_df is None:
                pairs_df     = reader.load('pairs')
                region_masks = define_hp_masks(pairs_df)
                for reg, msk in region_masks.items():
                    n = msk.sum()
                    print(f"    {reg}: {n} bipolar pairs" + (" ⚠ none!" if n == 0 else ""))

            evs = reader.load('task_events')

            # ---- find POINTING_FINISHED events -----------------------------
            pointing_evs = evs[evs['type'] == POINTING_EVENT_TYPE].copy()

            if len(pointing_evs) == 0:
                alt = evs[evs['type'].str.upper() == POINTING_EVENT_TYPE.upper()]
                if len(alt) > 0:
                    pointing_evs = alt.copy()
                    print(f"    Found {len(pointing_evs)} {pointing_evs['type'].iloc[0]} events (case variant)")
                else:
                    available = evs['type'].unique().tolist()
                    print(f"    ⚠ No POINTING_FINISHED events. Available types: {available}")
                    continue

            print(f"    Found {len(pointing_evs)} POINTING_FINISHED events")

            # ---- load EEG for all pointing events at once ------------------
            try:
                eeg_container = reader.load_eeg(
                    pointing_evs,
                    EPOCH_START_MS,
                    EPOCH_END_MS,
                    scheme=pairs_df
                )
            except Exception as e:
                print(f"    ✗ EEG load failed: {e}")
                continue

            eeg_data = eeg_container.data          # (n_events, n_channels, n_samples)
            sfreq    = float(eeg_container.samplerate)

            # Sample indices for the analysis window within the loaded epoch
            s0 = ms_to_sample(WIN_START_MS, EPOCH_START_MS, sfreq)
            s1 = ms_to_sample(WIN_END_MS,   EPOCH_START_MS, sfreq)

            print(f"    sfreq={sfreq:.0f} Hz | "
                  f"epoch samples: {eeg_data.shape[2]} | "
                  f"analysis window: [{s0}, {s1})")

            # ---- IRASA per event × region × frequency ----------------------
            pointing_evs_reset = pointing_evs.reset_index(drop=True)

            for ev_idx in range(len(pointing_evs_reset)):
                ev_row = pointing_evs_reset.iloc[ev_idx]

                trial       = ev_row.get('trial',          np.nan)
                eegoffset   = ev_row.get('eegoffset',      np.nan)
                item        = ev_row.get('item',           '')
                store_x     = ev_row.get('storeX',         np.nan)
                store_z     = ev_row.get('storeZ',         np.nan)
                inside_stim = ev_row.get('inside_stimuli', np.nan)

                eeg_epoch = eeg_data[ev_idx, :, s0:s1]  # (n_ch, n_win_samples)

                for region in REGIONS:
                    ch_indices = np.where(region_masks[region])[0]

                    frac_ssl_mean, osc_ssl_mean, n_valid_ch = irasa_for_event(
                        eeg_epoch, ch_indices, sfreq
                    )

                    for freq_idx, freq_hz in enumerate(IRASA_FREQS):
                        session_rows.append({
                            'experiment':   exp,
                            'subject':      subject,
                            'session':      session,
                            'event_idx':    ev_idx,
                            'trial':        trial,
                            'eegoffset':    eegoffset,
                            'item':         item,
                            'storeX':       store_x,
                            'storeZ':       store_z,
                            'inside_stim':  inside_stim,
                            'region':       region,
                            'n_channels':   n_valid_ch,
                            'freq_hz':      freq_hz,
                            'freq_idx':     freq_idx,
                            'frac_ssl':     frac_ssl_mean[freq_idx],
                            'osc_ssl':      osc_ssl_mean[freq_idx],
                        })

            # ---- save session CSV ------------------------------------------
            if session_rows:
                sess_df = pd.DataFrame(session_rows)
                sess_df.to_csv(fname, index=False)
                print(f"    ✓ Saved {len(session_rows):,} rows → {fname}")
                subject_rows.extend(session_rows)
            else:
                print(f"    ⚠ No rows produced for session {session}")

        except Exception as e:
            import traceback
            print(f"  ✗ [{exp}] {subject} session {session} FAILED: {e}")
            traceback.print_exc()
            continue

    return subject_rows

# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    whole_df = cml.CMLReader.get_data_index()

    all_rows = []

    for exp in EXPERIMENTS:
        exp_df   = whole_df.query('experiment == @exp')
        subjects = exp_df['subject'].unique()

        print(f"\n{'='*80}")
        print(f"Experiment: {exp}  |  {len(subjects)} subjects  |  n_jobs={N_JOBS}")
        print(f"{'='*80}")

        # --- parallel dispatch over subjects ---------------------------------
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
            delayed(process_subject)(subject, exp, exp_df, OUTPUT_DIR)
            for subject in subjects
        )

        # Flatten list-of-lists
        for subj_rows in results:
            all_rows.extend(subj_rows)

    # ============================================================================
    # MASTER CSV
    # ============================================================================
    print("\n" + "="*80)
    print("ALL DONE")
    print("="*80)

    if all_rows:
        master_df  = pd.DataFrame(all_rows)
        master_csv = os.path.join(OUTPUT_DIR, 'ALL_SUBJECTS_pointing_hp_irasa.csv')
        master_df.to_csv(master_csv, index=False)

        print(f"\n✓ Master CSV: {master_csv}")
        print(f"  Rows        : {len(master_df):,}")
        print(f"  Experiments : {master_df['experiment'].unique().tolist()}")
        print(f"  Subjects    : {master_df['subject'].nunique()}")
        print(f"  Sessions    : {master_df['session'].nunique()}")
        print(f"  Events      : {master_df.groupby(['experiment','subject','session','event_idx']).ngroups:,}")
        print(f"  Regions     : {sorted(master_df['region'].unique())}")
        print(f"  Frequencies : {master_df['freq_hz'].nunique()}")
        print(f"\nSample rows:")
        print(master_df.head(10).to_string(index=False))
    else:
        print("\n⚠ No data collected – check event type labels above.")

✓ IRASA imported successfully

Experiment: DBOY1  |  44 subjects  |  n_jobs=-1


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:  3.6min
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:  4.6min
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:  6.9min
[Parallel(n_jobs=-1)]: Done  33 tasks      | elapsed:  7.4min
[Parallel(n_jobs=-1)]: Done  42 out of  44 | elapsed:  9.1min remaining:   26.1s
[Parallel(n_jobs=-1)]: Done  44 out of  44 | elapsed:  9.3min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.



Experiment: EFRCourierReadOnly  |  8 subjects  |  n_jobs=-1


[Parallel(n_jobs=-1)]: Batch computation too fast (0.0835s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   2 out of   8 | elapsed:   49.1s remaining:  2.5min
[Parallel(n_jobs=-1)]: Done   3 out of   8 | elapsed:  1.2min remaining:  2.0min
[Parallel(n_jobs=-1)]: Done   4 out of   8 | elapsed:  1.7min remaining:  1.7min
[Parallel(n_jobs=-1)]: Done   5 out of   8 | elapsed:  1.8min remaining:  1.1min
[Parallel(n_jobs=-1)]: Done   6 out of   8 | elapsed:  3.8min remaining:  1.3min



ALL DONE

✓ Master CSV: ./irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa.csv
  Rows        : 11,772
  Experiments : ['DBOY1', 'EFRCourierReadOnly']
  Subjects    : 50
  Sessions    : 8
  Events      : 327
  Regions     : ['LHP', 'RHP']
  Frequencies : 18

Sample rows:
experiment subject  session  event_idx  trial  eegoffset item    storeX   storeZ  inside_stim region  n_channels  freq_hz  freq_idx  frac_ssl  osc_ssl
     DBOY1  R1494D        0          0      0    1560955 -999 14.601562 -58.8125          NaN    LHP           0     3.00         0       NaN      NaN
     DBOY1  R1494D        0          0      0    1560955 -999 14.601562 -58.8125          NaN    LHP           0     3.74         1       NaN      NaN
     DBOY1  R1494D        0          0      0    1560955 -999 14.601562 -58.8125          NaN    LHP           0     4.67         2       NaN      NaN
     DBOY1  R1494D        0          0      0    1560955 -999 14.601562 -58.8125          NaN    LHP           0   

[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:  4.7min remaining:    0.0s
[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:  4.7min finished


In [15]:

reader = cml.CMLReader('R1494D', 'DBOY1', session=0)
evs = reader.load('task_events')
evs['type'].unique()

array(['STORE_FAM', 'store mappings', 'pointing begins',
       'pointing finished', 'WORD', 'REC_START', 'REC_WORD', 'REC_STOP',
       'CUED_REC_CUE', 'CUED_REC_WORD_VV', 'CUED_REC_STOP',
       'CUED_REC_WORD', 'SR_START', 'SR_REC_WORD', 'SR_STOP', 'FFR_START',
       'FFR_REC_WORD', 'FFR_STOP'], dtype=object)

In [22]:
df = pd.read_csv('irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa.csv')


In [8]:
"""
Post-processing: average IRASA features across events AND sessions
for each subject × experiment × region × frequency.

Input : ALL_SUBJECTS_pointing_hp_irasa.csv   (trial-level master CSV)
Output: ALL_SUBJECTS_pointing_hp_irasa_subj_avg.csv
"""

import os
import pandas as pd
import numpy as np

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_CSV  = './irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa.csv'
OUTPUT_CSV = './irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_subj_avg.csv'

# ============================================================================
# LOAD
# ============================================================================

print(f"Loading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)
print(f"  Rows loaded : {len(df):,}")
print(f"  Subjects    : {df['subject'].nunique()}")
print(f"  Experiments : {df['experiment'].unique().tolist()}")
print(f"  Sessions    : {df['session'].nunique()}")
print(f"  Regions     : {sorted(df['region'].unique())}")
print(f"  Frequencies : {df['freq_hz'].nunique()}")

# ============================================================================
# STEP 1: average across events within each subject × session × region × freq
# (session-level mean)
# ============================================================================

print("\nStep 1: session-level averages ...")
session_avg = (
    df
    .groupby(
        ['experiment', 'subject', 'session', 'region', 'freq_hz', 'freq_idx'],
        sort=True, observed=True
    )
    .agg(
        n_events   = ('event_idx', 'nunique'),
        n_channels = ('n_channels', 'mean'),
        frac_ssl   = ('frac_ssl',   'mean'),
        osc_ssl    = ('osc_ssl',    'mean'),
    )
    .reset_index()
)
print(f"  Session-level rows: {len(session_avg):,}")

# ============================================================================
# STEP 2: average across sessions within each subject × region × freq
# (subject-level mean — each session contributes equally regardless of
#  how many events it contained, because we average the session means)
# ============================================================================

print("Step 2: subject-level averages (averaging over sessions) ...")
subj_avg = (
    session_avg
    .groupby(
        ['experiment', 'subject', 'region', 'freq_hz', 'freq_idx'],
        sort=True, observed=True
    )
    .agg(
        n_sessions  = ('session',    'nunique'),
        n_events    = ('n_events',   'sum'),      # total events across all sessions
        n_channels  = ('n_channels', 'mean'),     # mean valid channels (diagnostic)
        frac_ssl    = ('frac_ssl',   'mean'),
        osc_ssl     = ('osc_ssl',    'mean'),
    )
    .reset_index()
)

# ============================================================================
# SAVE
# ============================================================================

os.makedirs(os.path.dirname(OUTPUT_CSV) or '.', exist_ok=True)
subj_avg.to_csv(OUTPUT_CSV, index=False)

print(f"\n✓ Saved subject-average CSV: {OUTPUT_CSV}")
print(f"  Rows : {len(subj_avg):,}  "
      f"(= {subj_avg['subject'].nunique()} subj × "
      f"{subj_avg['region'].nunique()} regions × "
      f"{subj_avg['freq_hz'].nunique()} freqs)")
print(f"\nSample rows:")
print(subj_avg.head(12).to_string(index=False))

Loading ./irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa.csv ...
  Rows loaded : 11,772
  Subjects    : 50
  Experiments : ['DBOY1', 'EFRCourierReadOnly']
  Sessions    : 8
  Regions     : ['LHP', 'RHP']
  Frequencies : 18

Step 1: session-level averages ...
  Session-level rows: 3,816
Step 2: subject-level averages (averaging over sessions) ...

✓ Saved subject-average CSV: ./irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_subj_avg.csv
  Rows : 1,800  (= 50 subj × 2 regions × 18 freqs)

Sample rows:
experiment subject region  freq_hz  freq_idx  n_sessions  n_events  n_channels  frac_ssl  osc_ssl
     DBOY1  R1494D    LHP     3.00         0           3        15         0.0       NaN      NaN
     DBOY1  R1494D    LHP     3.74         1           3        15         0.0       NaN      NaN
     DBOY1  R1494D    LHP     4.67         2           3        15         0.0       NaN      NaN
     DBOY1  R1494D    LHP     5.82         3           3        15         0.0   

In [9]:
"""
Merge clustering scores (from ALL_SUBJECTS_irasa_per_freq.csv) with the
subject-averaged pointing IRASA output (ALL_SUBJECTS_pointing_hp_irasa_subj_avg.csv).

Steps
-----
1. Load the DBOY1 trial-level clustering CSV.
2. Average SP_clustering_from_prev and T_clustering_from_prev across all
   recalled words and trials → one score per subject (behavioral measure).
   Two-step: word-level → trial-level mean, then trial-level → subject-level mean
   so each trial contributes equally regardless of number of recalls.
3. Load the subject-averaged pointing IRASA CSV (DBOY1 + EFRCourierReadOnly).
4. Keep only DBOY1 subjects.
5. Merge: assign each subject's SP / TC score to every row (region × frequency)
   in the pointing IRASA table.  Within-subject the value is constant across
   region/frequency, exactly as requested.
6. Save merged CSV.
"""

import os
import pandas as pd
import numpy as np

# ============================================================================
# CONFIGURATION  – adjust paths if needed
# ============================================================================

CLUSTERING_CSV   = 'ALL_SUBJECTS_irasa_clean.csv'
POINTING_AVG_CSV = './irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_subj_avg.csv'
OUTPUT_CSV       = './irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_with_clustering.csv'

# ============================================================================
# STEP 1: Load clustering data
# ============================================================================

print(f"Loading clustering CSV: {CLUSTERING_CSV}")
clust_df = pd.read_csv(CLUSTERING_CSV)

print(f"  Rows          : {len(clust_df):,}")
print(f"  Subjects      : {clust_df['subject'].nunique()}")
print(f"  Columns       : {clust_df.columns.tolist()}")

# ============================================================================
# STEP 2: Subject-level clustering averages (two-step, from_prev only)
#
# Because clustering scores are per recalled-word × region × frequency but
# are purely behavioral, we first deduplicate to the word level
# (subject × session × trial × recalled_word) before averaging, so that
# the region/frequency repetitions don't inflate the mean.
# ============================================================================

print("\nComputing subject-level clustering averages...")

# --- deduplicate to one row per recalled word --------------------------------
word_level = (
    clust_df
    .drop_duplicates(subset=['subject', 'session', 'trial', 'recalled_word'])
    [['subject', 'session', 'trial', 'recalled_word',
      'SP_clustering_from_prev', 'T_clustering_from_prev']]
    .copy()
)

print(f"  Unique recalled-word rows : {len(word_level):,}")

# --- Step 2a: trial-level mean (each trial contributes equally) -------------
trial_avg = (
    word_level
    .groupby(['subject', 'session', 'trial'], sort=True)
    .agg(
        SP_clustering = ('SP_clustering_from_prev', 'mean'),
        TC_clustering = ('T_clustering_from_prev',  'mean'),
        n_recalls     = ('recalled_word',            'count'),
    )
    .reset_index()
)

print(f"  Trial-level rows          : {len(trial_avg):,}")

# --- Step 2b: session-level mean (each session contributes equally) ---------
session_avg = (
    trial_avg
    .groupby(['subject', 'session'], sort=True)
    .agg(
        SP_clustering = ('SP_clustering', 'mean'),
        TC_clustering = ('TC_clustering', 'mean'),
        n_trials      = ('trial',         'nunique'),
    )
    .reset_index()
)

# --- Step 2c: subject-level mean (each session contributes equally) ---------
subj_clust = (
    session_avg
    .groupby('subject', sort=True)
    .agg(
        SP_clustering = ('SP_clustering', 'mean'),
        TC_clustering = ('TC_clustering', 'mean'),
        n_sessions    = ('session',       'nunique'),
        n_trials      = ('n_trials',      'sum'),
    )
    .reset_index()
)

print(f"\nSubject-level clustering summary ({len(subj_clust)} subjects):")
print(subj_clust.to_string(index=False))

# ============================================================================
# STEP 3: Load pointing IRASA subject-average CSV
# ============================================================================

print(f"\nLoading pointing IRASA CSV: {POINTING_AVG_CSV}")
pointing_df = pd.read_csv(POINTING_AVG_CSV)

print(f"  Rows          : {len(pointing_df):,}")
print(f"  Experiments   : {pointing_df['experiment'].unique().tolist()}")
print(f"  Subjects      : {pointing_df['subject'].nunique()}")

# ============================================================================
# STEP 4: Keep only DBOY1 subjects
# ============================================================================

pointing_df = pointing_df.copy()
print(f"\nAfter keeping DBOY1 only:")
print(f"  Rows     : {len(pointing_df):,}")
print(f"  Subjects : {pointing_df['subject'].nunique()}")

# ============================================================================
# STEP 5: Merge clustering scores onto pointing IRASA rows
# Each subject gets the same SP/TC score across all region × frequency rows.
# ============================================================================

print("\nMerging clustering scores onto pointing IRASA table...")

merged_df = pointing_df.merge(
    subj_clust[['subject', 'SP_clustering', 'TC_clustering']],
    on='subject',
    how='left'
)

n_matched   = merged_df['SP_clustering'].notna().sum()
n_unmatched = merged_df['SP_clustering'].isna().sum()
print(f"  Rows with clustering score : {n_matched:,}")
print(f"  Rows without match (NaN)   : {n_unmatched:,}")

if n_unmatched > 0:
    missing_subjs = merged_df.loc[merged_df['SP_clustering'].isna(), 'subject'].unique()
    print(f"  Subjects without clustering data: {missing_subjs.tolist()}")

# ============================================================================
# STEP 6: Save
# ============================================================================

os.makedirs(os.path.dirname(OUTPUT_CSV) or '.', exist_ok=True)
merged_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✓ Saved merged CSV: {OUTPUT_CSV}")
print(f"  Rows       : {len(merged_df):,}  "
      f"(= {merged_df['subject'].nunique()} subj × "
      f"{merged_df['region'].nunique()} regions × "
      f"{merged_df['freq_hz'].nunique()} freqs)")
print(f"  Columns    : {merged_df.columns.tolist()}")
print(f"\nSample rows:")
print(merged_df.head(12).to_string(index=False))

Loading clustering CSV: ALL_SUBJECTS_irasa_clean.csv
  Rows          : 28,836
  Subjects      : 36
  Columns       : ['subject', 'session', 'trial', 'recalled_word', 'region', 'freq_hz', 'freq_idx', 'SP_clustering_from_prev', 'SP_clustering_to_next', 'T_clustering_from_prev', 'T_clustering_to_next', 'recall_stim', 'encoding_stim', 'frequency', 'encoding_frac_ssl', 'encoding_osc_ssl', 'retrieval_frac_ssl', 'retrieval_osc_ssl']

Computing subject-level clustering averages...
  Unique recalled-word rows : 1,094
  Trial-level rows          : 217

Subject-level clustering summary (36 subjects):
subject  SP_clustering  TC_clustering  n_sessions  n_trials
 R1494D       0.641435       0.637384           2         7
 R1501J       0.500060       0.515253           2        12
 R1502D       0.510194       0.653410           2        12
 R1503E       0.519154       0.565685           3        14
 R1504E       0.521346       0.643213           1         4
 R1509E       0.567220       0.611352      

In [10]:
"""
Between-Subject LMM Analysis — Pointing IRASA × Clustering
===========================================================

Mirrors the between-subject Level-2 OLS structure from the gamma-band
iEEG analysis, adapted for the pointing IRASA data.

Input
-----
  ALL_SUBJECTS_pointing_hp_irasa_with_clustering.csv
    Columns: experiment, subject, region, freq_hz, freq_idx,
             n_events, n_channels, frac_ssl, osc_ssl,
             SP_clustering, TC_clustering

Model structure
---------------
Two separate between-subject OLS models (one per IRASA component):

  COMPONENT: osc_ssl  |  frac_ssl
  
  Step 1 — Average over freq_hz within subject × region
           → one row per subject × region (× clust_T after stacking)

  Step 2 — Compute between-subject neural mean per subject
           (collapsed over region, used as the neural predictor)

  Step 3 — Stack SP and TC clustering long, add clust_T dummy

  Step 4 — Aggregate to subject × clust_T × region means

  OLS formula (same as reference script):
    clustering_score ~ neural_between + clust_T + region_RHP
                     + neural_between:clust_T
                     + neural_between:region_RHP

  Step 5 — BH-FDR correction across all tests within each component model

Output
------
  pointing_between_lmm_results.csv   — full results table
  pointing_between_lmm_report.txt    — formatted text report
  pointing_between_lmm_figure.png    — scatter + regression figure (2 panels)
"""

import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_CSV  = './irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_with_clustering.csv'
OUTPUT_DIR = './irasa_pointing_finished_hp'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# LOAD
# ============================================================================

print(f"Loading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)

print(f"  Rows       : {len(df):,}")
print(f"  Subjects   : {df['subject'].nunique()}")
print(f"  Regions    : {sorted(df['region'].unique())}")
print(f"  Freq bins  : {df['freq_hz'].nunique()}")
print(f"  Columns    : {df.columns.tolist()}")

# Sanity check — drop rows missing clustering or neural data
df = df.dropna(subset=['SP_clustering', 'TC_clustering', 'frac_ssl', 'osc_ssl']).copy()
print(f"  Rows after dropna: {len(df):,}")

# ============================================================================
# EFFECT SIZE HELPERS  (identical to reference script)
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# FDR HELPER  (identical to reference script)
# ============================================================================

def apply_fdr(results, p_cols):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p    = np.array(all_p, dtype=float)
    valid    = ~np.isnan(all_p)
    fdr_p    = np.full_like(all_p, np.nan)
    if valid.sum() > 0:
        _, p_corr, _, _ = multipletests(all_p[valid], method='fdr_bh')
        fdr_p[valid]    = p_corr
    idx = 0
    for pc in p_cols:
        n            = len(results)
        col_fdr      = pc.replace('p_', 'pfdr_')
        col_sig      = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

# ============================================================================
# MAIN LOOP — one iteration per IRASA component
# ============================================================================

COMPONENTS = ['osc', 'frac']
results_rows = []

# We'll collect plot data for the figure
plot_data_store = {}

for comp in COMPONENTS:

    col_ssl = f'{comp}_ssl'   # osc_ssl  or  frac_ssl

    # ------------------------------------------------------------------
    # Step 1: average over freq_hz → one row per subject × region
    # (freq_hz is a nuisance dimension here; between model does not need it —
    #  exactly as the reference script drops freq_hz before Level-2 OLS)
    # ------------------------------------------------------------------
    subj_region_avg = (
        df
        .groupby(['subject', 'region'])
        [[col_ssl, 'SP_clustering', 'TC_clustering']]
        .mean()
        .reset_index()
    )

    # ------------------------------------------------------------------
    # Step 2: between-subject neural mean (collapsed over region)
    # ------------------------------------------------------------------
    subj_mean_map = subj_region_avg.groupby('subject')[col_ssl].mean()
    subj_region_avg[col_ssl + '_between'] = subj_region_avg['subject'].map(subj_mean_map)

    # ------------------------------------------------------------------
    # Step 3: stack SP and TC long, add clust_T dummy
    # ------------------------------------------------------------------
    sp_df = subj_region_avg.copy()
    sp_df['clustering_score'] = sp_df['SP_clustering']
    sp_df['clustering_type']  = 'SP'

    tc_df = subj_region_avg.copy()
    tc_df['clustering_score'] = tc_df['TC_clustering']
    tc_df['clustering_type']  = 'TC'

    long_df = pd.concat([sp_df, tc_df], ignore_index=True)
    long_df['clust_T']    = (long_df['clustering_type'] == 'TC').astype(int)
    long_df['region_RHP'] = (long_df['region'] == 'RHP').astype(int)

    col_between = col_ssl + '_between'

    # ------------------------------------------------------------------
    # Step 4: aggregate to subject × clust_T × region cell means
    # (mirrors reference: between_data = long_avg.groupby([subject, clust_T, region_RHP]).mean())
    # ------------------------------------------------------------------
    between_data = (
        long_df
        .groupby(['subject', 'clust_T', 'region_RHP'])
        [[col_between, 'clustering_score']]
        .mean()
        .reset_index()
        .dropna()
    )

    # store for plotting
    plot_data_store[comp] = {
        'long_df':      long_df,
        'between_data': between_data,
        'col_between':  col_between,
    }

    # ------------------------------------------------------------------
    # Step 5: Between-subject OLS  (identical formula to reference)
    # ------------------------------------------------------------------
    try:
        formula = (
            f'clustering_score ~ {col_between} + clust_T + region_RHP '
            f'+ {col_between}:clust_T '
            f'+ {col_between}:region_RHP'
        )
        res        = smf.ols(formula, data=between_data).fit()
        df_resid   = res.df_resid
        n_subj     = between_data['subject'].nunique()
        n_obs      = int(res.nobs)

        def g(key):
            t = res.tvalues.get(key, np.nan)
            p = res.pvalues.get(key, np.nan)
            e = partial_eta_sq(t, df_resid)
            return t, p, e

        t_neural,   p_neural,   e_neural   = g(col_between)
        t_xclust,   p_xclust,   e_xclust   = g(f'{col_between}:clust_T')
        t_xregion,  p_xregion,  e_xregion  = g(f'{col_between}:region_RHP')
        t_clust,    p_clust,    e_clust    = g('clust_T')
        t_region,   p_region,   e_region   = g('region_RHP')
        d_neural    = cohens_d_from_t(t_neural, n_subj)
        note        = 'ok'

    except Exception as exc:
        import traceback; traceback.print_exc()
        (t_neural, p_neural, e_neural,
         t_xclust,  p_xclust,  e_xclust,
         t_xregion, p_xregion, e_xregion,
         t_clust,   p_clust,   e_clust,
         t_region,  p_region,  e_region,
         d_neural,  n_subj,    n_obs) = [np.nan] * 18
        note = f'failed: {exc}'

    results_rows.append({
        'component':          comp,
        # neural main
        't_neural':           t_neural,  'p_neural':           p_neural,  'eta2p_neural':           e_neural,
        'd_neural':           d_neural,
        # clustering type main
        't_clustT':           t_clust,   'p_clustT':           p_clust,   'eta2p_clustT':           e_clust,
        # region main
        't_region':           t_region,  'p_region':           p_region,  'eta2p_region':           e_region,
        # neural × clust_T
        't_neural_x_clustT':  t_xclust,  'p_neural_x_clustT':  p_xclust,  'eta2p_neural_x_clustT':  e_xclust,
        # neural × region
        't_neural_x_region':  t_xregion, 'p_neural_x_region':  p_xregion, 'eta2p_neural_x_region':  e_xregion,
        # metadata
        'n_subjects':         n_subj,    'n_obs':              n_obs,     'note':                   note,
    })

    print(f"\n[{comp.upper()}] OLS fit: {note}  |  n_subj={n_subj}  n_obs={n_obs}")

# ============================================================================
# FDR CORRECTION — applied across both component models jointly
# ============================================================================

results = pd.DataFrame(results_rows)

p_cols = ['p_neural', 'p_clustT', 'p_region',
          'p_neural_x_clustT', 'p_neural_x_region']

results = apply_fdr(results, p_cols)

# Uncorrected significance flags
for pc in p_cols:
    results['sig_unc_' + pc[2:]] = results[pc].apply(
        lambda x: x < 0.05 if pd.notna(x) else False)

# Save results table
results_csv = os.path.join(OUTPUT_DIR, 'pointing_between_lmm_results.csv')
results.to_csv(results_csv, index=False)
print(f"\n✓ Results saved: {results_csv}")

# ============================================================================
# FIGURE  — scatter + regression, 1×2 panel (one panel per component)
# Mirrors the reference script's visualization style
# ============================================================================

COL_SP   = '#2166AC'
COL_TC   = '#D6604D'
COL_GRID = '#E8E8E8'

fig = plt.figure(figsize=(12, 5))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.38)
panel_labels = ['A', 'B']

for ci, comp in enumerate(COMPONENTS):
    ax          = fig.add_subplot(gs[0, ci])
    col_between = plot_data_store[comp]['col_between']
    between_data= plot_data_store[comp]['between_data']

    sp_data = between_data[between_data['clust_T'] == 0].copy()
    tc_data = between_data[between_data['clust_T'] == 1].copy()

    # Scatter
    ax.scatter(sp_data[col_between], sp_data['clustering_score'],
               color=COL_SP, s=60, zorder=3, alpha=0.85,
               label='SP clustering', edgecolors='white', linewidths=0.5)
    ax.scatter(tc_data[col_between], tc_data['clustering_score'],
               color=COL_TC, s=60, zorder=3, alpha=0.85,
               label='TC clustering', edgecolors='white', linewidths=0.5)

    # Regression lines
    for dat, col in [(sp_data, COL_SP), (tc_data, COL_TC)]:
        x = dat[col_between].values
        y = dat['clustering_score'].values
        if len(x) > 1 and not np.all(np.isnan(x)):
            m, b = np.polyfit(x, y, 1)
            xr   = np.linspace(np.nanmin(x), np.nanmax(x), 100)
            ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

    # Annotation
    row = results[results['component'] == comp].iloc[0]

    def pstr(p):
        if pd.isna(p):    return 'n.s.'
        if p < 0.001:     return 'p<.001'
        if p < 0.05:      return f'p={p:.3f}'
        return f'p={p:.2f} n.s.'

    ann = (f"Neural main: t={row['t_neural']:+.2f}, {pstr(row['pfdr_neural'])}, "
           f"d={row['d_neural']:.2f}\n"
           f"× Clust type: t={row['t_neural_x_clustT']:+.2f}, "
           f"{pstr(row['pfdr_neural_x_clustT'])}")
    ax.text(0.03, 0.97, ann, transform=ax.transAxes,
            fontsize=8, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7',
                      edgecolor='#CCCCCC', alpha=0.9))

    ax.text(-0.12, 1.05, panel_labels[ci], transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')

    comp_label = 'Oscillatory' if comp == 'osc' else 'Aperiodic (Fractal)'
    ax.set_title(f'Pointing — {comp_label} power', fontsize=11,
                 fontweight='bold', pad=8)
    ax.set_xlabel('IRASA power (subject mean, SSL)', fontsize=9)
    ax.set_ylabel('Clustering score', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, color=COL_GRID, linewidth=0.7, zorder=0)
    ax.tick_params(labelsize=8)

    if ci == 0:
        ax.legend(fontsize=8.5, framealpha=0.9, loc='lower right')

fig.suptitle(
    'Between-subject: Post-pointing hippocampal power predicts clustering\n'
    f'(iEEG, n={results["n_subjects"].max():.0f} DBOY1 subjects)',
    fontsize=12, fontweight='bold', y=1.02
)

fig_path = os.path.join(OUTPUT_DIR, 'pointing_between_lmm_figure.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✓ Figure saved: {fig_path}")

# ============================================================================
# TEXT REPORT
# ============================================================================

lines = []
L = lines.append

L("=" * 110)
L("POINTING IRASA — Between-Subject OLS Analysis × Clustering (SP & TC)")
L("  Neural predictors : osc_ssl_between, frac_ssl_between (subject means, averaged over freq_hz)")
L("  DVs               : SP_clustering and TC_clustering (stacked long, clust_T dummy)")
L("  Factors           : clust_T (SP=0 / TC=1), region_RHP (LHP=0 / RHP=1)")
L("  FDR               : Benjamini-Hochberg, applied jointly across both component models")
L("  Sig               :  * uncorrected p<.05  |  † FDR-corrected p<.05")
L("=" * 110)

HDR = f"  {'Effect':<40} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>9}  sig"
SEP = f"  {'─' * 95}"

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*'  if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†'  if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc)  else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr)  else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)   else '    NaN'
    ds  = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return f"  {label:<40} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {ds:>9}  {sig_unc}{sig_fdr}"

for _, row in results.iterrows():
    comp = row['component']
    L(f"\n{'━' * 110}")
    L(f"  COMPONENT: {'OSCILLATORY (OSC)' if comp == 'osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"  n_subjects={int(row['n_subjects'])}   n_obs={int(row['n_obs'])}   status={row['note']}")
    L(f"{'━' * 110}")

    if row['note'] == 'ok':
        L(HDR); L(SEP)
        L(fmt('Neural main effect',
              row['t_neural'],          row['p_neural'],         row['pfdr_neural'],         row['eta2p_neural'],         row['d_neural']))
        L(fmt('Clustering type (TC vs SP)',
              row['t_clustT'],          row['p_clustT'],         row['pfdr_clustT'],         row['eta2p_clustT']))
        L(fmt('Region (RHP vs LHP)',
              row['t_region'],          row['p_region'],         row['pfdr_region'],         row['eta2p_region']))
        L(fmt('Neural × clustering_type',
              row['t_neural_x_clustT'], row['p_neural_x_clustT'],row['pfdr_neural_x_clustT'],row['eta2p_neural_x_clustT']))
        L(fmt('Neural × region',
              row['t_neural_x_region'], row['p_neural_x_region'],row['pfdr_neural_x_region'],row['eta2p_neural_x_region']))
    else:
        L(f"  *** MODEL FAILED: {row['note']} ***")

L(f"\n\n{'=' * 110}")
L("FDR-SIGNIFICANT EFFECTS:")
found = False
for _, row in results.iterrows():
    comp_label = 'OSC' if row['component'] == 'osc' else 'FRAC'
    for pc, label in [('pfdr_neural',          'Neural main'),
                      ('pfdr_clustT',          'Clustering type main'),
                      ('pfdr_region',          'Region main'),
                      ('pfdr_neural_x_clustT', 'Neural × clust_T'),
                      ('pfdr_neural_x_region', 'Neural × region')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            found = True
            L(f"  [{comp_label}]  {label:<35}  p_fdr={p:.3f}")
if not found:
    L("  None.")

L(f"\n{'=' * 110}")
L("FILES SAVED:")
L(f"  {results_csv}")
L(f"  {fig_path}")
L(f"  {os.path.join(OUTPUT_DIR, 'pointing_between_lmm_report.txt')}")
L("=" * 110)

report = '\n'.join(lines)
print(report)

report_path = os.path.join(OUTPUT_DIR, 'pointing_between_lmm_report.txt')
with open(report_path, 'w') as f:
    f.write(report)

print("\nDone.")

Loading ./irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_with_clustering.csv ...
  Rows       : 1,800
  Subjects   : 50
  Regions    : ['LHP', 'RHP']
  Freq bins  : 18
  Columns    : ['experiment', 'subject', 'region', 'freq_hz', 'freq_idx', 'n_sessions', 'n_events', 'n_channels', 'frac_ssl', 'osc_ssl', 'SP_clustering', 'TC_clustering']
  Rows after dropna: 972

[OSC] OLS fit: ok  |  n_subj=36  n_obs=108

[FRAC] OLS fit: ok  |  n_subj=36  n_obs=108

✓ Results saved: ./irasa_pointing_finished_hp/pointing_between_lmm_results.csv
✓ Figure saved: ./irasa_pointing_finished_hp/pointing_between_lmm_figure.png
POINTING IRASA — Between-Subject OLS Analysis × Clustering (SP & TC)
  Neural predictors : osc_ssl_between, frac_ssl_between (subject means, averaged over freq_hz)
  DVs               : SP_clustering and TC_clustering (stacked long, clust_T dummy)
  Factors           : clust_T (SP=0 / TC=1), region_RHP (LHP=0 / RHP=1)
  FDR               : Benjamini-Hochberg, applied jointly a

In [11]:
"""
Merge clustering scores with task and resting-state IRASA into a single
long-format DataFrame for three-way OLS analysis.

Structure of output
-------------------
One row per subject × clustering_type × neural_state (task / rest)

  - Clustering scores: three-step average (word → trial → session → subject)
    same value repeated across neural_state rows for each subject
  - Neural SSL: subject-level mean from task epoch OR resting epoch
  - neural_state: 'task' / 'rest'

Paths
-----
  CLUSTERING_CSV : ALL_SUBJECTS_irasa_clean.csv  (behavioral + task neural)
  TASK_IRASA_CSV : ALL_SUBJECTS_irasa_clean.csv  (same file, task neural SSL)
  REST_IRASA_CSV : ./irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_subj_avg.csv
  OUTPUT_CSV     : ./irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_with_clustering_stacked.csv
"""

import os
import pandas as pd
import numpy as np

# ============================================================================
# CONFIGURATION
# ============================================================================

CLUSTERING_CSV = 'ALL_SUBJECTS_irasa_clean.csv'
TASK_IRASA_CSV = 'ALL_SUBJECTS_irasa_clean.csv'
REST_IRASA_CSV = './irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_subj_avg.csv'
OUTPUT_CSV     = './irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_with_clustering_stacked.csv'

# ============================================================================
# STEP 1: Load clustering CSV
# ============================================================================

print(f"Loading clustering CSV: {CLUSTERING_CSV}")
clust_df = pd.read_csv(CLUSTERING_CSV)
print(f"  Rows     : {len(clust_df):,}")
print(f"  Subjects : {clust_df['subject'].nunique()}")
print(f"  Columns  : {clust_df.columns.tolist()}")

# ============================================================================
# STEP 2: Subject-level clustering averages
# Three-step: word → trial → session → subject
# ============================================================================

print("\nComputing subject-level clustering averages...")

# --- deduplicate to one row per recalled word --------------------------------
word_level = (
    clust_df
    .drop_duplicates(subset=['subject', 'session', 'trial', 'recalled_word'])
    [['subject', 'session', 'trial', 'recalled_word',
      'SP_clustering_from_prev', 'T_clustering_from_prev']]
    .copy()
)
print(f"  Unique recalled-word rows: {len(word_level):,}")

# --- Step 2a: word → trial --------------------------------------------------
trial_avg = (
    word_level
    .groupby(['subject', 'session', 'trial'], sort=True)
    .agg(
        SP_clustering = ('SP_clustering_from_prev', 'mean'),
        TC_clustering = ('T_clustering_from_prev',  'mean'),
        n_recalls     = ('recalled_word',            'count'),
    )
    .reset_index()
)
print(f"  Trial-level rows : {len(trial_avg):,}")

# --- Step 2b: trial → session -----------------------------------------------
session_avg = (
    trial_avg
    .groupby(['subject', 'session'], sort=True)
    .agg(
        SP_clustering = ('SP_clustering', 'mean'),
        TC_clustering = ('TC_clustering', 'mean'),
        n_trials      = ('trial',         'nunique'),
    )
    .reset_index()
)
print(f"  Session-level rows: {len(session_avg):,}")

# --- Step 2c: session → subject ---------------------------------------------
subj_clust = (
    session_avg
    .groupby('subject', sort=True)
    .agg(
        SP_clustering = ('SP_clustering', 'mean'),
        TC_clustering = ('TC_clustering', 'mean'),
        n_sessions    = ('session',       'nunique'),
        n_trials      = ('n_trials',      'sum'),
    )
    .reset_index()
)

print(f"\nSubject-level clustering summary ({len(subj_clust)} subjects):")
print(subj_clust.to_string(index=False))

# ============================================================================
# STEP 3: Stack clustering into long format
# One row per subject × clustering_type
# ============================================================================

sp_df = subj_clust[['subject', 'SP_clustering', 'n_sessions', 'n_trials']].copy()
sp_df['clustering_score'] = sp_df['SP_clustering']
sp_df['clustering_type']  = 'SP'

tc_df = subj_clust[['subject', 'TC_clustering', 'n_sessions', 'n_trials']].copy()
tc_df['clustering_score'] = tc_df['TC_clustering']
tc_df['clustering_type']  = 'TC'

clust_long = pd.concat([sp_df, tc_df], ignore_index=True)
clust_long = clust_long[['subject', 'clustering_type', 'clustering_score',
                          'n_sessions', 'n_trials']].copy()

print(f"\nClustering long format: {len(clust_long)} rows "
      f"({clust_long['subject'].nunique()} subjects × 2 types)")

# ============================================================================
# STEP 4: Neural SSL averaging functions
# ============================================================================

def subject_neural_avg_from_clean(df, label):
    """
    For ALL_SUBJECTS_irasa_clean.csv: trial-level data.
    Three-step average matching the clustering hierarchy:
    word → trial → session → subject.
    Uses retrieval SSL columns if present,
    otherwise falls back to frac_ssl / osc_ssl directly.
    """
    print(f"\n{label} — computing three-step subject neural averages...")

    # Determine which SSL columns to use
    if 'retrieval_frac_ssl' in df.columns and 'retrieval_osc_ssl' in df.columns:
        frac_col = 'retrieval_frac_ssl'
        osc_col  = 'retrieval_osc_ssl'
        print(f"  Using retrieval SSL columns: {frac_col}, {osc_col}")
    else:
        frac_col = 'frac_ssl'
        osc_col  = 'osc_ssl'
        print(f"  Using generic SSL columns: {frac_col}, {osc_col}")

    # Deduplicate to word level (drop region × freq repetitions)
    word_level = (
        df
        .drop_duplicates(subset=['subject', 'session', 'trial', 'recalled_word'])
        [['subject', 'session', 'trial', 'recalled_word', frac_col, osc_col]]
        .copy()
    )
    print(f"  Unique recalled-word rows: {len(word_level):,}")

    # Step a: word → trial
    trial_avg = (
        word_level
        .groupby(['subject', 'session', 'trial'])
        [[frac_col, osc_col]]
        .mean()
        .reset_index()
    )

    # Step b: trial → session
    session_avg = (
        trial_avg
        .groupby(['subject', 'session'])
        [[frac_col, osc_col]]
        .mean()
        .reset_index()
    )

    # Step c: session → subject
    subj_avg = (
        session_avg
        .groupby('subject')
        [[frac_col, osc_col]]
        .mean()
        .reset_index()
        .rename(columns={frac_col: 'frac_ssl', osc_col: 'osc_ssl'})
    )

    print(f"  Subject-level rows: {len(subj_avg)}")
    print(subj_avg[['frac_ssl', 'osc_ssl']].describe().to_string())
    return subj_avg


def subject_neural_avg_from_pointing(df, label):
    """
    For ALL_SUBJECTS_pointing_hp_irasa_subj_avg.csv: already event-averaged.
    Just average over region × freq_hz → one row per subject.
    """
    print(f"\n{label} — averaging over region × freq_hz...")
    subj_avg = (
        df
        .groupby('subject')
        [['frac_ssl', 'osc_ssl']]
        .mean()
        .reset_index()
    )
    print(f"  Subject-level rows: {len(subj_avg)}")
    print(subj_avg[['frac_ssl', 'osc_ssl']].describe().to_string())
    return subj_avg


# ============================================================================
# STEP 5: Load task and rest CSVs, filter to DBOY1, compute neural averages
# ============================================================================

def load_and_filter_dboy1(path, label):
    print(f"\nLoading {label} CSV: {path}")
    df = pd.read_csv(path)
    print(f"  Rows     : {len(df):,}")
    print(f"  Subjects : {df['subject'].nunique()}")
    if 'experiment' in df.columns:
        df = df[df['experiment'] == 'DBOY1'].copy()
        print(f"  After DBOY1 filter — Rows: {len(df):,} | "
              f"Subjects: {df['subject'].nunique()}")
    else:
        print("  WARNING: no 'experiment' column found, keeping all subjects")
    return df

# Task: ALL_SUBJECTS_irasa_clean.csv (already loaded as clust_df, just reuse)
task_df = clust_df.copy()
if 'experiment' in task_df.columns:
    task_df = task_df[task_df['experiment'] == 'DBOY1'].copy()
    print(f"\nTask DBOY1 filter — Rows: {len(task_df):,} | "
          f"Subjects: {task_df['subject'].nunique()}")

# Rest: pointing IRASA CSV
rest_df = load_and_filter_dboy1(REST_IRASA_CSV, 'Rest')

# Compute subject-level neural averages
task_neural = subject_neural_avg_from_clean(task_df,    'Task')
rest_neural = subject_neural_avg_from_pointing(rest_df, 'Rest')

# ============================================================================
# STEP 6: Tag with neural_state and stack
# ============================================================================

task_neural['neural_state'] = 'task'
rest_neural['neural_state'] = 'rest'

neural_stacked = pd.concat([task_neural, rest_neural], ignore_index=True)

print(f"\nStacked neural rows: {len(neural_stacked)}")
print(f"neural_state counts:\n"
      f"{neural_stacked['neural_state'].value_counts().to_string()}")

# ============================================================================
# STEP 7: Merge clustering long × neural stacked
# Each subject gets 4 rows:
#   SP × task | SP × rest | TC × task | TC × rest
# ============================================================================

merged_df = clust_long.merge(
    neural_stacked,
    on='subject',
    how='inner'
)

print(f"\nAfter merge:")
print(f"  Rows     : {len(merged_df):,}  "
      f"(expect {merged_df['subject'].nunique()} subj × 2 types × 2 states = "
      f"{merged_df['subject'].nunique() * 4})")
print(f"  Subjects : {merged_df['subject'].nunique()}")

# Report any subjects lost in the inner join
clust_subjs  = set(clust_long['subject'].unique())
neural_subjs = set(neural_stacked['subject'].unique())
only_clust   = clust_subjs  - neural_subjs
only_neural  = neural_subjs - clust_subjs
if only_clust:
    print(f"  ⚠ Subjects with clustering but no neural data : {only_clust}")
if only_neural:
    print(f"  ⚠ Subjects with neural data but no clustering : {only_neural}")

# ============================================================================
# STEP 8: Add OLS dummy columns
# ============================================================================

merged_df['clust_T']           = (merged_df['clustering_type'] == 'TC').astype(int)
merged_df['neural_state_task'] = (merged_df['neural_state']    == 'task').astype(int)

# Final column order
final_cols = [
    'subject',
    'clustering_type', 'clust_T',
    'clustering_score',
    'n_sessions', 'n_trials',
    'neural_state', 'neural_state_task',
    'frac_ssl', 'osc_ssl',
]
merged_df = (
    merged_df[final_cols]
    .sort_values(['subject', 'clustering_type', 'neural_state'])
    .reset_index(drop=True)
)

# ============================================================================
# STEP 9: Save
# ============================================================================

os.makedirs(os.path.dirname(OUTPUT_CSV) or '.', exist_ok=True)
merged_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✓ Saved: {OUTPUT_CSV}")
print(f"  Rows     : {len(merged_df):,}")
print(f"  Subjects : {merged_df['subject'].nunique()}")
print(f"  Columns  : {merged_df.columns.tolist()}")

print(f"\nRow counts by clustering_type × neural_state:")
print(merged_df.groupby(['clustering_type', 'neural_state'])
      .size().reset_index(name='n_rows').to_string(index=False))

print(f"\nSample rows (first 8):")
print(merged_df.head(8).to_string(index=False))

Loading clustering CSV: ALL_SUBJECTS_irasa_clean.csv
  Rows     : 28,836
  Subjects : 36
  Columns  : ['subject', 'session', 'trial', 'recalled_word', 'region', 'freq_hz', 'freq_idx', 'SP_clustering_from_prev', 'SP_clustering_to_next', 'T_clustering_from_prev', 'T_clustering_to_next', 'recall_stim', 'encoding_stim', 'frequency', 'encoding_frac_ssl', 'encoding_osc_ssl', 'retrieval_frac_ssl', 'retrieval_osc_ssl']

Computing subject-level clustering averages...
  Unique recalled-word rows: 1,094
  Trial-level rows : 217
  Session-level rows: 76

Subject-level clustering summary (36 subjects):
subject  SP_clustering  TC_clustering  n_sessions  n_trials
 R1494D       0.641435       0.637384           2         7
 R1501J       0.500060       0.515253           2        12
 R1502D       0.510194       0.653410           2        12
 R1503E       0.519154       0.565685           3        14
 R1504E       0.521346       0.643213           1         4
 R1509E       0.567220       0.611352      

In [12]:
"""
Between-Subject OLS Analysis — Three-Way Interaction
=====================================================
Neural × Clustering Type × Neural State

Input
-----
  ALL_SUBJECTS_pointing_hp_irasa_with_clustering_stacked.csv
    Columns: subject, clustering_type, clust_T, clustering_score,
             n_sessions, n_trials, neural_state, neural_state_task,
             frac_ssl, osc_ssl

  One row per subject × clustering_type × neural_state (4 rows per subject)
  Neural SSL is already subject-averaged (no region/freq dims to collapse)

Output
------
  pointing_3way_ols_results.csv
  pointing_3way_ols_report.txt
  pointing_3way_ols_figure.png
"""

import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_CSV  = './irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_with_clustering_stacked.csv'
OUTPUT_DIR = './irasa_pointing_finished_hp'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# LOAD
# ============================================================================

print(f"Loading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)

print(f"  Rows         : {len(df):,}")
print(f"  Subjects     : {df['subject'].nunique()}")
print(f"  Columns      : {df.columns.tolist()}")
print(f"  Neural states: {df['neural_state'].unique().tolist()}")
print(f"  Clust types  : {df['clustering_type'].unique().tolist()}")

# Sanity check — should be exactly 4 rows per subject
rows_per_subj = df.groupby('subject').size()
if (rows_per_subj != 4).any():
    print(f"  ⚠ Some subjects do not have exactly 4 rows:")
    print(rows_per_subj[rows_per_subj != 4].to_string())
else:
    print(f"  ✓ All subjects have exactly 4 rows (SP×task, SP×rest, TC×task, TC×rest)")

df = df.dropna(subset=['clustering_score', 'frac_ssl', 'osc_ssl']).copy()
print(f"  Rows after dropna: {len(df):,}")

for state in ['task', 'rest']:
    n = df[df['neural_state'] == state]['subject'].nunique()
    print(f"  neural_state='{state}': {n} subjects")

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# FDR HELPER
# ============================================================================

def apply_fdr(results, p_cols):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p = np.array(all_p, dtype=float)
    valid = ~np.isnan(all_p)
    fdr_p = np.full_like(all_p, np.nan)
    if valid.sum() > 0:
        _, p_corr, _, _ = multipletests(all_p[valid], method='fdr_bh')
        fdr_p[valid]    = p_corr
    idx = 0
    for pc in p_cols:
        n                        = len(results)
        col_fdr                  = pc.replace('p_', 'pfdr_')
        col_sig                  = pc.replace('p_', 'sig_fdr_')
        results[col_fdr]         = fdr_p[idx:idx+n]
        results[col_sig]         = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

# ============================================================================
# MAIN LOOP — one iteration per IRASA component
# ============================================================================

COMPONENTS      = ['osc', 'frac']
results_rows    = []
plot_data_store = {}

for comp in COMPONENTS:

    col_ssl     = f'{comp}_ssl'
    col_between = f'{col_ssl}_between'

    # ------------------------------------------------------------------
    # The input CSV already has one row per subject × clust_type × state.
    # Neural SSL is already subject-averaged — no region/freq to collapse.
    # Just rename the ssl column to 'col_between' to match OLS formula,
    # and use the data directly as the between_data table.
    # ------------------------------------------------------------------

    between_data = df[['subject', 'clust_T', 'neural_state_task',
                        'clustering_score', col_ssl]].copy()
    between_data = between_data.rename(columns={col_ssl: col_between})
    between_data = between_data.dropna().reset_index(drop=True)

    n_subj = between_data['subject'].nunique()
    n_obs  = len(between_data)
    print(f"\n[{comp.upper()}] n_subjects={n_subj} | n_obs={n_obs} | "
          f"state counts: {between_data['neural_state_task'].value_counts().to_dict()}")

    plot_data_store[comp] = {
        'between_data': between_data,
        'col_between':  col_between,
    }

    # ------------------------------------------------------------------
    # Three-way OLS
    # Note: region_RHP dropped — not present in this merged CSV.
    # KEY TERM: col_between:clust_T:neural_state_task
    # ------------------------------------------------------------------
    try:
        formula = (
            f'clustering_score ~ '
            f'{col_between} '
            f'+ clust_T '
            f'+ neural_state_task '
            f'+ {col_between}:clust_T '
            f'+ {col_between}:neural_state_task '
            f'+ clust_T:neural_state_task '
            f'+ {col_between}:clust_T:neural_state_task'
        )
        res      = smf.ols(formula, data=between_data).fit()
        df_resid = res.df_resid
        n_obs    = int(res.nobs)

        def g(key):
            t = res.tvalues.get(key, np.nan)
            p = res.pvalues.get(key, np.nan)
            e = partial_eta_sq(t, df_resid)
            return t, p, e

        # Main effects
        t_neural,   p_neural,   e_neural   = g(col_between)
        t_clustT,   p_clustT,   e_clustT   = g('clust_T')
        t_state,    p_state,    e_state    = g('neural_state_task')
        # Two-way interactions
        t_nx_clust, p_nx_clust, e_nx_clust = g(f'{col_between}:clust_T')
        t_nx_state, p_nx_state, e_nx_state = g(f'{col_between}:neural_state_task')
        t_cx_state, p_cx_state, e_cx_state = g('clust_T:neural_state_task')
        # Three-way — KEY TEST
        t_3way,     p_3way,     e_3way     = g(f'{col_between}:clust_T:neural_state_task')

        d_neural = cohens_d_from_t(t_neural, n_subj)
        d_3way   = cohens_d_from_t(t_3way,   n_subj)
        note     = 'ok'

    except Exception as exc:
        import traceback; traceback.print_exc()
        (t_neural,   p_neural,   e_neural,
         t_clustT,   p_clustT,   e_clustT,
         t_state,    p_state,    e_state,
         t_nx_clust, p_nx_clust, e_nx_clust,
         t_nx_state, p_nx_state, e_nx_state,
         t_cx_state, p_cx_state, e_cx_state,
         t_3way,     p_3way,     e_3way,
         d_neural,   d_3way,
         n_subj,     n_obs) = [np.nan] * 26
        note = f'failed: {exc}'

    results_rows.append({
        'component':         comp,
        # main effects
        't_neural':          t_neural,   'p_neural':          p_neural,   'eta2p_neural':          e_neural,   'd_neural':  d_neural,
        't_clustT':          t_clustT,   'p_clustT':          p_clustT,   'eta2p_clustT':          e_clustT,
        't_state':           t_state,    'p_state':           p_state,    'eta2p_state':           e_state,
        # two-way interactions
        't_neural_x_clustT': t_nx_clust, 'p_neural_x_clustT': p_nx_clust, 'eta2p_neural_x_clustT': e_nx_clust,
        't_neural_x_state':  t_nx_state, 'p_neural_x_state':  p_nx_state, 'eta2p_neural_x_state':  e_nx_state,
        't_clustT_x_state':  t_cx_state, 'p_clustT_x_state':  p_cx_state, 'eta2p_clustT_x_state':  e_cx_state,
        # three-way (KEY)
        't_3way':            t_3way,     'p_3way':            p_3way,     'eta2p_3way':            e_3way,     'd_3way': d_3way,
        # metadata
        'n_subjects':        n_subj,     'n_obs':             n_obs,      'note':                  note,
    })

    print(f"  THREE-WAY (Neural × Clust × State): t={t_3way:+.3f}, p={p_3way:.3f}, η²p={e_3way:.3f}")

# ============================================================================
# FDR CORRECTION — across both components jointly
# ============================================================================

results = pd.DataFrame(results_rows)

p_cols = [
    'p_neural', 'p_clustT', 'p_state',
    'p_neural_x_clustT', 'p_neural_x_state', 'p_clustT_x_state',
    'p_3way'
]
results = apply_fdr(results, p_cols)

results_csv = os.path.join(OUTPUT_DIR, 'pointing_3way_ols_results.csv')
results.to_csv(results_csv, index=False)
print(f"\n✓ Results saved: {results_csv}")

# ============================================================================
# FIGURE — 2×2 grid: rows = component (osc/frac), cols = state (task/rest)
# ============================================================================

COL_SP   = '#2166AC'
COL_TC   = '#D6604D'
COL_GRID = '#E8E8E8'

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.patch.set_facecolor('white')
panel_labels = [['A', 'B'], ['C', 'D']]
state_labels = {1: 'Task', 0: 'Rest'}
comp_labels  = {'osc': 'Oscillatory', 'frac': 'Aperiodic (Fractal)'}

def pstr(p):
    if pd.isna(p):  return 'n.s.'
    if p < 0.001:   return 'p<.001'
    if p < 0.05:    return f'p={p:.3f}'
    return f'p={p:.2f} n.s.'

for ri, comp in enumerate(COMPONENTS):
    col_between  = plot_data_store[comp]['col_between']
    between_data = plot_data_store[comp]['between_data']
    row_results  = results[results['component'] == comp].iloc[0]

    for ci, state_val in enumerate([1, 0]):   # task left, rest right
        ax        = axes[ri][ci]
        state_dat = between_data[between_data['neural_state_task'] == state_val]

        sp_data = state_dat[state_dat['clust_T'] == 0]
        tc_data = state_dat[state_dat['clust_T'] == 1]

        ax.scatter(sp_data[col_between], sp_data['clustering_score'],
                   color=COL_SP, s=60, alpha=0.85, zorder=3,
                   label='SP clustering', edgecolors='white', linewidths=0.5)
        ax.scatter(tc_data[col_between], tc_data['clustering_score'],
                   color=COL_TC, s=60, alpha=0.85, zorder=3,
                   label='TC clustering', edgecolors='white', linewidths=0.5)

        for dat, col in [(sp_data, COL_SP), (tc_data, COL_TC)]:
            x = dat[col_between].values
            y = dat['clustering_score'].values
            if len(x) > 1 and not np.all(np.isnan(x)):
                m, b = np.polyfit(x, y, 1)
                xr   = np.linspace(np.nanmin(x), np.nanmax(x), 100)
                ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

        ax.text(-0.12, 1.07, panel_labels[ri][ci],
                transform=ax.transAxes, fontsize=13, fontweight='bold', va='top')
        ax.set_title(f'{comp_labels[comp]} — {state_labels[state_val]}',
                     fontsize=10, fontweight='bold', pad=6)
        ax.set_xlabel('IRASA power (subject mean, SSL)', fontsize=8.5)
        ax.set_ylabel('Clustering score', fontsize=8.5)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, linewidth=0.6, zorder=0)
        ax.tick_params(labelsize=8)
        if ri == 0 and ci == 0:
            ax.legend(fontsize=8, framealpha=0.9, loc='lower right')

    # Annotate 3-way stats on the task panel of each row
    ann = (f"3-way (N×C×State): t={row_results['t_3way']:+.2f}, "
           f"{pstr(row_results['pfdr_3way'])}, η²p={row_results['eta2p_3way']:.3f}\n"
           f"2-way N×C (task+rest pooled): t={row_results['t_neural_x_clustT']:+.2f}, "
           f"{pstr(row_results['pfdr_neural_x_clustT'])}")
    axes[ri][0].text(0.03, 0.97, ann,
                     transform=axes[ri][0].transAxes,
                     fontsize=7.5, va='top', ha='left',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7',
                               edgecolor='#CCCCCC', alpha=0.9))

fig.suptitle(
    'Three-way interaction: Hippocampal gamma × Clustering type × Neural state\n'
    f'(iEEG, n={int(results["n_subjects"].max())} DBOY1 subjects)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'pointing_3way_ols_figure.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✓ Figure saved: {fig_path}")

# ============================================================================
# TEXT REPORT
# ============================================================================

lines = []
L = lines.append

L("=" * 115)
L("THREE-WAY BETWEEN-SUBJECT OLS: Neural × Clustering Type × Neural State")
L("  Neural predictor  : subject-mean osc_ssl / frac_ssl")
L("  DV                : clustering_score (SP and TC stacked, clust_T dummy)")
L("  Factors           : clust_T (SP=0 / TC=1)")
L("                      neural_state_task (rest=0 / task=1)")
L("  KEY TEST          : neural:clust_T:neural_state_task")
L("                      → does Neural × Clust interaction differ between task and rest?")
L("  FDR               : Benjamini-Hochberg across all 7 p-values × 2 components")
L("  Sig               : * uncorr p<.05   † FDR-corr p<.05")
L("=" * 115)

HDR = f"  {'Effect':<50} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>8}  sig"
SEP = "  " + "─" * 100

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*' if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc)  else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr)  else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)   else '    NaN'
    ds  = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return f"  {label:<50} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {ds:>8}  {sig_unc}{sig_fdr}"

for _, row in results.iterrows():
    comp = row['component']
    L(f"\n{'━' * 115}")
    L(f"  COMPONENT: {'OSCILLATORY (OSC)' if comp == 'osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"  n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_obs'])}  status={row['note']}")
    L(f"{'━' * 115}")

    if row['note'] == 'ok':
        L(HDR); L(SEP)
        L("  — Main effects —")
        L(fmt('Neural (between-subject)',
              row['t_neural'],     row['p_neural'],     row['pfdr_neural'],     row['eta2p_neural'],     row['d_neural']))
        L(fmt('Clustering type (TC vs SP)',
              row['t_clustT'],     row['p_clustT'],     row['pfdr_clustT'],     row['eta2p_clustT']))
        L(fmt('Neural state (task vs rest)',
              row['t_state'],      row['p_state'],      row['pfdr_state'],      row['eta2p_state']))
        L("  — Two-way interactions —")
        L(fmt('Neural × clustering type',
              row['t_neural_x_clustT'],  row['p_neural_x_clustT'],  row['pfdr_neural_x_clustT'],  row['eta2p_neural_x_clustT']))
        L(fmt('Neural × neural state',
              row['t_neural_x_state'],   row['p_neural_x_state'],   row['pfdr_neural_x_state'],   row['eta2p_neural_x_state']))
        L(fmt('Clustering type × neural state',
              row['t_clustT_x_state'],   row['p_clustT_x_state'],   row['pfdr_clustT_x_state'],   row['eta2p_clustT_x_state']))
        L("  — Three-way interaction (KEY TEST) —")
        L(fmt('Neural × clustering type × state ⭐',
              row['t_3way'], row['p_3way'], row['pfdr_3way'], row['eta2p_3way'], row['d_3way']))
    else:
        L(f"  *** MODEL FAILED: {row['note']} ***")

L(f"\n\n{'=' * 115}")
L("FDR-SIGNIFICANT EFFECTS:")
found = False
for _, row in results.iterrows():
    comp_label = 'OSC' if row['component'] == 'osc' else 'FRAC'
    for pc, label in [
        ('pfdr_neural',          'Neural main'),
        ('pfdr_clustT',          'Clustering type main'),
        ('pfdr_state',           'Neural state main'),
        ('pfdr_neural_x_clustT', 'Neural × Clust type'),
        ('pfdr_neural_x_state',  'Neural × State'),
        ('pfdr_clustT_x_state',  'Clust type × State'),
        ('pfdr_3way',            'Neural × Clust × State (THREE-WAY) ⭐'),
    ]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            found = True
            L(f"  [{comp_label}]  {label:<45}  p_fdr={p:.3f}")
if not found:
    L("  None.")

L(f"\n{'=' * 115}")
L("FILES SAVED:")
L(f"  {results_csv}")
L(f"  {fig_path}")
L(f"  {os.path.join(OUTPUT_DIR, 'pointing_3way_ols_report.txt')}")
L("=" * 115)

report = '\n'.join(lines)
print(report)

with open(os.path.join(OUTPUT_DIR, 'pointing_3way_ols_report.txt'), 'w') as f:
    f.write(report)

print("\nDone.")

Loading ./irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_with_clustering_stacked.csv ...
  Rows         : 134
  Subjects     : 36
  Columns      : ['subject', 'clustering_type', 'clust_T', 'clustering_score', 'n_sessions', 'n_trials', 'neural_state', 'neural_state_task', 'frac_ssl', 'osc_ssl']
  Neural states: ['rest', 'task']
  Clust types  : ['SP', 'TC']
  ⚠ Some subjects do not have exactly 4 rows:
subject
R1620J    2
R1637T    2
R1642J    2
R1651J    2
R1653J    2
  Rows after dropna: 134
  neural_state='task': 36 subjects
  neural_state='rest': 31 subjects

[OSC] n_subjects=36 | n_obs=134 | state counts: {1: 72, 0: 62}
  THREE-WAY (Neural × Clust × State): t=+0.830, p=0.408, η²p=0.005

[FRAC] n_subjects=36 | n_obs=134 | state counts: {1: 72, 0: 62}
  THREE-WAY (Neural × Clust × State): t=+0.247, p=0.806, η²p=0.000

✓ Results saved: ./irasa_pointing_finished_hp/pointing_3way_ols_results.csv
✓ Figure saved: ./irasa_pointing_finished_hp/pointing_3way_ols_figure.png
THREE-W

In [13]:
"""
Between-Subject OLS Analysis — Two-Way Interaction
====================================================
Gamma power × Neural state → Spatial clustering score only

Input
-----
  ALL_SUBJECTS_pointing_hp_irasa_with_clustering_stacked.csv
    Columns: subject, clustering_type, clust_T, clustering_score,
             n_sessions, n_trials, neural_state, neural_state_task,
             frac_ssl, osc_ssl

  Filtered to SP clustering only (clust_T == 0).
  One row per subject × neural_state (2 rows per subject).

Output
------
  pointing_2way_ols_spatial_results.csv
  pointing_2way_ols_spatial_report.txt
  pointing_2way_ols_spatial_figure.png
"""

import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_CSV  = './irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_with_clustering_stacked.csv'
OUTPUT_DIR = './irasa_pointing_finished_hp'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# LOAD & FILTER TO SPATIAL CLUSTERING ONLY
# ============================================================================

print(f"Loading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)

print(f"  Rows (all)   : {len(df):,}")
print(f"  Subjects     : {df['subject'].nunique()}")

# Keep spatial clustering only
df = df[df['clust_T'] == 0].copy()
print(f"  Rows (SP only): {len(df):,}")
print(f"  Subjects (SP) : {df['subject'].nunique()}")
print(f"  Neural states : {df['neural_state'].unique().tolist()}")

# Sanity check — should be exactly 2 rows per subject (task + rest)
rows_per_subj = df.groupby('subject').size()
if (rows_per_subj != 2).any():
    print(f"  ⚠ Some subjects do not have exactly 2 rows:")
    print(rows_per_subj[rows_per_subj != 2].to_string())
else:
    print(f"  ✓ All subjects have exactly 2 rows (SP×task, SP×rest)")

df = df.dropna(subset=['clustering_score', 'frac_ssl', 'osc_ssl']).copy()
print(f"  Rows after dropna: {len(df):,}")

for state in ['task', 'rest']:
    n = df[df['neural_state'] == state]['subject'].nunique()
    print(f"  neural_state='{state}': {n} subjects")

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# FDR HELPER
# ============================================================================

def apply_fdr(results, p_cols):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p = np.array(all_p, dtype=float)
    valid = ~np.isnan(all_p)
    fdr_p = np.full_like(all_p, np.nan)
    if valid.sum() > 0:
        _, p_corr, _, _ = multipletests(all_p[valid], method='fdr_bh')
        fdr_p[valid] = p_corr
    idx = 0
    for pc in p_cols:
        n            = len(results)
        col_fdr      = pc.replace('p_', 'pfdr_')
        col_sig      = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

# ============================================================================
# MAIN LOOP — one iteration per IRASA component (osc / frac)
# ============================================================================

COMPONENTS      = ['osc', 'frac']
results_rows    = []
plot_data_store = {}

for comp in COMPONENTS:

    col_ssl     = f'{comp}_ssl'
    col_between = f'{col_ssl}_between'

    between_data = df[['subject', 'neural_state_task',
                        'clustering_score', col_ssl]].copy()
    between_data = between_data.rename(columns={col_ssl: col_between})
    between_data = between_data.dropna().reset_index(drop=True)

    n_subj = between_data['subject'].nunique()
    n_obs  = len(between_data)
    print(f"\n[{comp.upper()}] n_subjects={n_subj} | n_obs={n_obs} | "
          f"state counts: {between_data['neural_state_task'].value_counts().to_dict()}")

    plot_data_store[comp] = {
        'between_data': between_data,
        'col_between':  col_between,
    }

    # ------------------------------------------------------------------
    # Two-way OLS
    # DV         : clustering_score (spatial only)
    # Predictors : gamma power (osc_ssl / frac_ssl)
    #              neural_state_task (task=1 / rest=0)
    #              interaction: gamma × neural_state_task
    # KEY TERM   : col_between:neural_state_task
    # ------------------------------------------------------------------
    try:
        formula = (
            f'clustering_score ~ '
            f'{col_between} '
            f'+ neural_state_task '
            f'+ {col_between}:neural_state_task'
        )
        res      = smf.ols(formula, data=between_data).fit()
        df_resid = res.df_resid
        n_obs    = int(res.nobs)

        def g(key):
            t = res.tvalues.get(key, np.nan)
            p = res.pvalues.get(key, np.nan)
            e = partial_eta_sq(t, df_resid)
            return t, p, e

        t_neural,   p_neural,   e_neural   = g(col_between)
        t_state,    p_state,    e_state    = g('neural_state_task')
        t_2way,     p_2way,     e_2way     = g(f'{col_between}:neural_state_task')

        d_neural = cohens_d_from_t(t_neural, n_subj)
        d_2way   = cohens_d_from_t(t_2way,   n_subj)
        note     = 'ok'

    except Exception as exc:
        import traceback; traceback.print_exc()
        (t_neural, p_neural, e_neural,
         t_state,  p_state,  e_state,
         t_2way,   p_2way,   e_2way,
         d_neural, d_2way,
         n_subj,   n_obs) = [np.nan] * 13
        note = f'failed: {exc}'

    results_rows.append({
        'component':        comp,
        # main effects
        't_neural':         t_neural,  'p_neural':         p_neural,  'eta2p_neural':         e_neural,  'd_neural': d_neural,
        't_state':          t_state,   'p_state':          p_state,   'eta2p_state':           e_state,
        # two-way interaction (KEY)
        't_2way':           t_2way,    'p_2way':           p_2way,    'eta2p_2way':            e_2way,    'd_2way':   d_2way,
        # metadata
        'n_subjects':       n_subj,    'n_obs':            n_obs,     'note':                  note,
    })

    print(f"  MAIN  gamma        : t={t_neural:+.3f}, p={p_neural:.3f}, η²p={e_neural:.3f}")
    print(f"  MAIN  state        : t={t_state:+.3f},  p={p_state:.3f},  η²p={e_state:.3f}")
    print(f"  2-WAY gamma×state  : t={t_2way:+.3f},   p={p_2way:.3f},   η²p={e_2way:.3f}  ← KEY")

# ============================================================================
# FDR CORRECTION — across both components jointly
# ============================================================================

results = pd.DataFrame(results_rows)

p_cols = ['p_neural', 'p_state', 'p_2way']
results = apply_fdr(results, p_cols)

results_csv = os.path.join(OUTPUT_DIR, 'pointing_2way_ols_spatial_results.csv')
results.to_csv(results_csv, index=False)
print(f"\n✓ Results saved: {results_csv}")

# ============================================================================
# FIGURE — 1×2: columns = component (osc / frac)
# Each panel: scatter of gamma power vs spatial clustering,
#             task and rest in different colors with regression lines
# ============================================================================

COL_TASK = '#2166AC'   # blue  — task
COL_REST = '#D6604D'   # red   — rest
COL_GRID = '#E8E8E8'

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('white')
panel_labels = ['A', 'B']
comp_labels  = {'osc': 'Oscillatory (gamma)', 'frac': 'Aperiodic (Fractal)'}
state_labels = {1: 'Task', 0: 'Rest'}
state_colors = {1: COL_TASK, 0: COL_REST}

def pstr(p):
    if pd.isna(p):  return 'n.s.'
    if p < 0.001:   return 'p<.001'
    if p < 0.05:    return f'p={p:.3f}'
    return f'p={p:.2f} n.s.'

for ci, comp in enumerate(COMPONENTS):
    col_between  = plot_data_store[comp]['col_between']
    between_data = plot_data_store[comp]['between_data']
    row_results  = results[results['component'] == comp].iloc[0]
    ax           = axes[ci]

    for state_val in [1, 0]:
        dat   = between_data[between_data['neural_state_task'] == state_val]
        color = state_colors[state_val]
        label = state_labels[state_val]

        ax.scatter(dat[col_between], dat['clustering_score'],
                   color=color, s=65, alpha=0.85, zorder=3,
                   label=label, edgecolors='white', linewidths=0.5)

        x = dat[col_between].values
        y = dat['clustering_score'].values
        if len(x) > 1 and not np.all(np.isnan(x)):
            m, b = np.polyfit(x, y, 1)
            xr   = np.linspace(np.nanmin(x), np.nanmax(x), 100)
            ax.plot(xr, m*xr + b, color=color, linewidth=2.2, alpha=0.9)

    # Stats annotation
    ann = (
        f"Main gamma : t={row_results['t_neural']:+.2f}, {pstr(row_results['pfdr_neural'])}, η²p={row_results['eta2p_neural']:.3f}\n"
        f"Main state : t={row_results['t_state']:+.2f},  {pstr(row_results['pfdr_state'])},  η²p={row_results['eta2p_state']:.3f}\n"
        f"Interaction: t={row_results['t_2way']:+.2f},   {pstr(row_results['pfdr_2way'])},   η²p={row_results['eta2p_2way']:.3f}  ⭐"
    )
    ax.text(0.03, 0.97, ann,
            transform=ax.transAxes,
            fontsize=7.5, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7',
                      edgecolor='#CCCCCC', alpha=0.9))

    ax.text(-0.10, 1.05, panel_labels[ci],
            transform=ax.transAxes, fontsize=13, fontweight='bold', va='top')
    ax.set_title(f'{comp_labels[comp]}\nSpatial clustering score',
                 fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('IRASA power (subject mean, SSL)', fontsize=9)
    ax.set_ylabel('Spatial clustering score', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, color=COL_GRID, linewidth=0.6, zorder=0)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=9, framealpha=0.9, loc='lower right')

fig.suptitle(
    'Two-way interaction: Hippocampal gamma × Neural state → Spatial clustering\n'
    f'(iEEG, n={int(results["n_subjects"].max())} DBOY1 subjects, SP clustering only)',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'pointing_2way_ols_spatial_figure.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✓ Figure saved: {fig_path}")

# ============================================================================
# TEXT REPORT
# ============================================================================

lines = []
L = lines.append

L("=" * 100)
L("TWO-WAY BETWEEN-SUBJECT OLS: Gamma power × Neural state → Spatial clustering score")
L("  DV                : clustering_score (SP only, clust_T == 0)")
L("  Continuous pred   : osc_ssl / frac_ssl (subject-mean IRASA power)")
L("  Categorical pred  : neural_state_task (rest=0 / task=1)")
L("  KEY TEST          : gamma:neural_state_task")
L("                      → does gamma→spatial clustering slope differ task vs rest?")
L("  FDR               : Benjamini-Hochberg across 3 p-values × 2 components")
L("  Sig               : * uncorr p<.05   † FDR-corr p<.05")
L("=" * 100)

HDR = f"  {'Effect':<45} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>8}  sig"
SEP = "  " + "─" * 90

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*' if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc) else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr) else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)  else '    NaN'
    ds  = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return f"  {label:<45} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {ds:>8}  {sig_unc}{sig_fdr}"

for _, row in results.iterrows():
    comp = row['component']
    L(f"\n{'━' * 100}")
    L(f"  COMPONENT: {'OSCILLATORY (OSC)' if comp == 'osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"  n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_obs'])}  status={row['note']}")
    L(f"{'━' * 100}")

    if row['note'] == 'ok':
        L(HDR); L(SEP)
        L("  — Main effects —")
        L(fmt('Gamma power (between-subject)',
              row['t_neural'], row['p_neural'], row['pfdr_neural'], row['eta2p_neural'], row['d_neural']))
        L(fmt('Neural state (task vs rest)',
              row['t_state'],  row['p_state'],  row['pfdr_state'],  row['eta2p_state']))
        L("  — Two-way interaction (KEY TEST) —")
        L(fmt('Gamma power × neural state ⭐',
              row['t_2way'],   row['p_2way'],   row['pfdr_2way'],   row['eta2p_2way'],   row['d_2way']))
    else:
        L(f"  *** MODEL FAILED: {row['note']} ***")

L(f"\n\n{'=' * 100}")
L("FDR-SIGNIFICANT EFFECTS:")
found = False
for _, row in results.iterrows():
    comp_label = 'OSC' if row['component'] == 'osc' else 'FRAC'
    for pc, label in [
        ('pfdr_neural', 'Gamma power main effect'),
        ('pfdr_state',  'Neural state main effect'),
        ('pfdr_2way',   'Gamma × Neural state (TWO-WAY) ⭐'),
    ]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            found = True
            L(f"  [{comp_label}]  {label:<45}  p_fdr={p:.3f}")
if not found:
    L("  None.")

L(f"\n{'=' * 100}")
L("FILES SAVED:")
L(f"  {results_csv}")
L(f"  {fig_path}")
L(f"  {os.path.join(OUTPUT_DIR, 'pointing_2way_ols_spatial_report.txt')}")
L("=" * 100)

report = '\n'.join(lines)
print(report)

with open(os.path.join(OUTPUT_DIR, 'pointing_2way_ols_spatial_report.txt'), 'w') as f:
    f.write(report)

print("\nDone.")

Loading ./irasa_pointing_finished_hp/ALL_SUBJECTS_pointing_hp_irasa_with_clustering_stacked.csv ...
  Rows (all)   : 134
  Subjects     : 36
  Rows (SP only): 67
  Subjects (SP) : 36
  Neural states : ['rest', 'task']
  ⚠ Some subjects do not have exactly 2 rows:
subject
R1620J    1
R1637T    1
R1642J    1
R1651J    1
R1653J    1
  Rows after dropna: 67
  neural_state='task': 36 subjects
  neural_state='rest': 31 subjects

[OSC] n_subjects=36 | n_obs=67 | state counts: {1: 36, 0: 31}
  MAIN  gamma        : t=+0.892, p=0.376, η²p=0.012
  MAIN  state        : t=+0.673,  p=0.504,  η²p=0.007
  2-WAY gamma×state  : t=-0.928,   p=0.357,   η²p=0.013  ← KEY

[FRAC] n_subjects=36 | n_obs=67 | state counts: {1: 36, 0: 31}
  MAIN  gamma        : t=+0.676, p=0.502, η²p=0.007
  MAIN  state        : t=-0.183,  p=0.855,  η²p=0.001
  2-WAY gamma×state  : t=-0.299,   p=0.766,   η²p=0.001  ← KEY

✓ Results saved: ./irasa_pointing_finished_hp/pointing_2way_ols_spatial_results.csv
✓ Figure saved: ./irasa_

In [ ]:
"""
OLS: (task - rest) IRASA delta predicts SP / TC clustering
===========================================================
For each subject: delta = task_ssl - rest_ssl
Simple OLS per component × clustering type (4 models total):
  clustering_score ~ delta_neural
"""

import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

INPUT_CSV  = '/mnt/user-data/uploads/ALL_SUBJECTS_pointing_hp_irasa_with_clustering_stacked.csv'
OUTPUT_DIR = '/mnt/user-data/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
df = df.dropna(subset=['clustering_score', 'frac_ssl', 'osc_ssl'])

# ── Compute delta per subject × clustering_type ───────────────────────────────
rest = df[df['neural_state'] == 'rest'][['subject', 'clustering_type', 'clustering_score', 'osc_ssl', 'frac_ssl']].copy()
task = df[df['neural_state'] == 'task'][['subject', 'clustering_type', 'osc_ssl', 'frac_ssl']].copy()

merged = rest.merge(task, on=['subject', 'clustering_type'], suffixes=('_rest', '_task'))
merged['delta_osc']  = merged['osc_ssl_task']  - merged['osc_ssl_rest']
merged['delta_frac'] = merged['frac_ssl_task'] - merged['frac_ssl_rest']

print(f"Subjects with both rest & task: {merged['subject'].nunique()}")
print(merged[['subject','clustering_type','clustering_score','delta_osc','delta_frac']].head(8))

# ── Helpers ──────────────────────────────────────────────────────────────────
def partial_eta_sq(t, df_r):
    return np.nan if (pd.isna(t) or pd.isna(df_r)) else t**2 / (t**2 + df_r)

def cohens_d(t, n):
    return np.nan if (pd.isna(t) or not n) else t / np.sqrt(n)

# ── Run 4 models ─────────────────────────────────────────────────────────────
COMPONENTS   = [('osc', 'delta_osc', 'Oscillatory'), ('frac', 'delta_frac', 'Aperiodic (Fractal)')]
CLUST_TYPES  = [('SP', 0), ('TC', 1)]
results_rows = []
plot_store   = {}

for comp_key, col_delta, comp_label in COMPONENTS:
    for clust_name, clust_val in CLUST_TYPES:
        data = merged[merged['clustering_type'] == clust_name].copy().dropna(subset=[col_delta, 'clustering_score'])
        n    = len(data)

        try:
            res    = smf.ols(f'clustering_score ~ {col_delta}', data=data).fit()
            df_r   = res.df_resid
            t      = res.tvalues.get(col_delta, np.nan)
            p      = res.pvalues.get(col_delta, np.nan)
            eta2   = partial_eta_sq(t, df_r)
            r2     = res.rsquared
            d      = cohens_d(t, n)
            note   = 'ok'
            print(f"\n[{comp_key.upper()} × {clust_name}]  n={n}")
            print(res.summary())
        except Exception as exc:
            t = p = eta2 = r2 = d = np.nan; note = str(exc)

        results_rows.append(dict(
            component=comp_key, comp_label=comp_label,
            clustering_type=clust_name,
            t=t, p=p, eta2p=eta2, r2=r2, d=d, n=n, note=note
        ))
        plot_store[(comp_key, clust_name)] = (data[[col_delta, 'clustering_score']].copy(), col_delta)

# ── FDR across all 4 models ───────────────────────────────────────────────────
results = pd.DataFrame(results_rows)
all_p   = results['p'].values.astype(float)
valid   = ~np.isnan(all_p)
fdr_p   = np.full_like(all_p, np.nan)
if valid.sum():
    _, corr, _, _ = multipletests(all_p[valid], method='fdr_bh')
    fdr_p[valid]  = corr
results['p_fdr'] = fdr_p
results['sig']   = fdr_p < 0.05
results.to_csv(os.path.join(OUTPUT_DIR, 'pointing_delta_ols_results.csv'), index=False)

# ── Figure: 2×2 (component × clustering type) ────────────────────────────────
COL_SP   = '#2166AC'
COL_TC   = '#D6604D'
COL_GRID = '#E8E8E8'
COLORS   = {'SP': COL_SP, 'TC': COL_TC}

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.patch.set_facecolor('white')
panel_labels = [['A','B'],['C','D']]

for ri, (comp_key, col_delta, comp_label) in enumerate(COMPONENTS):
    for ci, (clust_name, _) in enumerate(CLUST_TYPES):
        ax  = axes[ri][ci]
        dat, col = plot_store[(comp_key, clust_name)]
        row = results[(results['component']==comp_key) & (results['clustering_type']==clust_name)].iloc[0]
        col_c = COLORS[clust_name]

        ax.scatter(dat[col], dat['clustering_score'],
                   color=col_c, s=65, alpha=0.85, zorder=3,
                   edgecolors='white', linewidths=0.5)

        x = dat[col].values; y = dat['clustering_score'].values
        if len(x) > 1 and not np.all(np.isnan(x)):
            m, b = np.polyfit(x, y, 1)
            xr = np.linspace(x.min(), x.max(), 100)
            ax.plot(xr, m*xr+b, color=col_c, lw=2, alpha=0.9)

        def pstr(p):
            if pd.isna(p): return 'n.s.'
            return 'p<.001' if p<.001 else (f'p={p:.3f}' if p<.05 else f'p={p:.2f} n.s.')

        ann = (f"t={row['t']:+.2f}\n"
               f"p_unc={pstr(row['p'])}\n"
               f"p_fdr={pstr(row['p_fdr'])}\n"
               f"R²={row['r2']:.3f}, d={row['d']:.2f}")
        ax.text(0.03, 0.97, ann, transform=ax.transAxes, fontsize=8.5, va='top',
                bbox=dict(boxstyle='round,pad=0.3', fc='#F7F7F7', ec='#CCCCCC', alpha=0.9))

        ax.text(-0.12, 1.06, panel_labels[ri][ci], transform=ax.transAxes,
                fontsize=13, fontweight='bold', va='top')
        ax.set_title(f'{comp_label} — {clust_name} clustering', fontsize=10, fontweight='bold', pad=6)
        ax.set_xlabel('Δ IRASA power (task − rest, SSL)', fontsize=9)
        ax.set_ylabel(f'{clust_name} clustering score', fontsize=9)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, lw=0.7, zorder=0)
        ax.tick_params(labelsize=8)
        ax.axvline(0, color='gray', lw=0.8, ls='--', alpha=0.5)

n_subj = merged['subject'].nunique()
fig.suptitle(f'Does task-vs-rest hippocampal activation predict clustering?\n'
             f'(iEEG, n={n_subj} subjects with both rest & task)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'pointing_delta_ols_figure.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"\n✓ Figure: {fig_path}")

# ── Report ────────────────────────────────────────────────────────────────────
lines = ["="*80,
         "POINTING IRASA — (task − rest) delta predicts clustering",
         "  Predictor: delta_osc / delta_frac = task_ssl − rest_ssl (per subject)",
         "  Models: 4 simple OLS (component × clustering type)",
         "  FDR: Benjamini-Hochberg across all 4 models",
         "="*80,
         f"  {'Model':<35} {'t':>7} {'p_unc':>7} {'p_fdr':>7} {'R²':>6} {'d':>7}  sig"]

for _, row in results.iterrows():
    label = f"{row['comp_label']} × {row['clustering_type']}"
    su = '*' if (pd.notna(row['p'])     and row['p']     < 0.05) else ''
    sf = '†' if (pd.notna(row['p_fdr']) and row['p_fdr'] < 0.05) else ''
    lines.append(
        f"  {label:<35} {row['t']:>+7.3f} {row['p']:>7.3f} {row['p_fdr']:>7.3f} "
        f"{row['r2']:>6.3f} {row['d']:>+7.3f}  {su}{sf}")

lines += ["\nFDR-SIGNIFICANT:", "  None." if not results['sig'].any() else ""]
for _, row in results[results['sig']].iterrows():
    lines.append(f"  [{row['component'].upper()} × {row['clustering_type']}]  p_fdr={row['p_fdr']:.3f}")
lines.append("="*80)

report = '\n'.join(lines)
print(report)
with open(os.path.join(OUTPUT_DIR, 'pointing_delta_ols_report.txt'), 'w') as f:
    f.write(report)
print("Done.")